In [ ]:
import re 

In [ ]:
class TeluguSpellChecker:
    def __init__(self, vocabulary, prefix_len=2):
        self.vocab_set = set(vocabulary)
        
        self.prefix_len = prefix_len
        self.groups = {}
        for word in vocabulary:
            key = word[:prefix_len]  
            if key not in self.groups:
                self.groups[key] = []
            self.groups[key].append(word)

    def levenshtein_distance(self, s1, s2):
        m, n = len(s1), len(s2)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1):
            dp[i][0] = i
        for j in range(n+1):
            dp[0][j] = j
        for i in range(1, m+1):
            for j in range(1, n+1):
                if s1[i-1] == s2[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        return dp[m][n]

    def suggest_corrections(self, word, max_suggestions=3, max_distance=2):
        # if word in self.vocab_set:
        #     return []
        key = word[:self.prefix_len]
        candidate_words = self.groups.get(key, [])

        suggestions = []
        for correct_word in candidate_words:
            dist = self.levenshtein_distance(word, correct_word)
            if dist <= max_distance:
                suggestions.append((correct_word, dist))
        
        suggestions.sort(key=lambda x: x[1])
        return [w for w, _ in suggestions[:max_suggestions]]

In [ ]:
vocab_file = "vocabulary.txt" 
with open(vocab_file, 'r', encoding='utf-8') as f:
    vocabulary = [line.strip() for line in f if line.strip()]

spell_checker = TeluguSpellChecker(vocabulary, prefix_len=2)

while True:
    word = input("\nEnter a Telugu word to check spelling (or type 'exit' to quit): ").strip()
    if word.lower() == 'exit':
        print("Exiting spell checker.")
        break
    
    suggestions = spell_checker.suggest_corrections(word, max_suggestions=5, max_distance=2)
    
    if not suggestions:
        if word in vocabulary:
            print(f"'{word}' is correctly spelled!")
        else:
            print(f"No close matches found for '{word}'")
    else:
        print(f"'{word}' might be misspelled. Suggestions:")
        for i, w in enumerate(suggestions, 1):
            print(f"{i}. {w}")
